# FastAPI Advanced Features

Covers file uploads with `UploadFile`, streaming responses, simple in-memory caching, cursor-based pagination, and Server-Sent Events (SSE).

In [ ]:
# !pip install fastapi httpx python-multipart

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.testclient import TestClient
import io

app = FastAPI()

# --- File upload with UploadFile ---
@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    content = await file.read()
    return {
        "filename": file.filename,
        "content_type": file.content_type,
        "size_bytes": len(content),
        "preview": content[:50].decode("utf-8", errors="replace")
    }

@app.post("/upload-multiple")
async def upload_multiple(files: list[UploadFile] = File(...)):
    results = []
    for f in files:
        content = await f.read()
        results.append({"filename": f.filename, "size": len(content)})
    return {"uploaded": results, "count": len(results)}

client = TestClient(app)

# Single file upload
fake_file = io.BytesIO(b"Hello, this is a test file content!")
r = client.post("/upload", files={"file": ("test.txt", fake_file, "text/plain")})
print("Single upload:", r.json())

# Multiple file upload
files = [
    ("files", ("a.txt", io.BytesIO(b"File A content"), "text/plain")),
    ("files", ("b.txt", io.BytesIO(b"File B content here"), "text/plain")),
]
r = client.post("/upload-multiple", files=files)
print("Multiple upload:", r.json())

In [ ]:
# --- Streaming response ---
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
import asyncio
import time

app2 = FastAPI()

def generate_csv_rows(n: int = 5):
    """Generator that yields CSV rows one at a time."""
    yield "id,name,value\n"
    for i in range(1, n + 1):
        yield f"{i},item_{i},{i * 10.5}\n"

async def generate_async_chunks(n: int = 5):
    """Async generator for streaming data."""
    for i in range(1, n + 1):
        yield f"chunk {i}\n".encode()
        await asyncio.sleep(0)  # Yield control

@app2.get("/stream/csv")
def stream_csv(rows: int = 5):
    return StreamingResponse(
        generate_csv_rows(rows),
        media_type="text/csv",
        headers={"Content-Disposition": "attachment; filename=data.csv"}
    )

@app2.get("/stream/chunks")
def stream_chunks(n: int = 5):
    return StreamingResponse(generate_async_chunks(n), media_type="text/plain")

client2 = TestClient(app2)

r = client2.get("/stream/csv?rows=3")
print("CSV stream:")
print(r.text)
print("Content-Disposition:", r.headers.get("content-disposition"))

r = client2.get("/stream/chunks?n=4")
print("Chunks stream:")
print(r.text)

In [ ]:
# --- Simple in-memory caching ---
from fastapi import FastAPI
from fastapi.testclient import TestClient
from functools import lru_cache
from datetime import datetime, timedelta
import time

app3 = FastAPI()

# TTL cache implementation
class TTLCache:
    def __init__(self, ttl_seconds: int = 60):
        self._cache: dict = {}
        self._ttl = ttl_seconds

    def get(self, key: str):
        if key in self._cache:
            value, expires_at = self._cache[key]
            if datetime.now() < expires_at:
                return value
            del self._cache[key]
        return None

    def set(self, key: str, value):
        self._cache[key] = (value, datetime.now() + timedelta(seconds=self._ttl))

    def invalidate(self, key: str):
        self._cache.pop(key, None)

cache = TTLCache(ttl_seconds=5)
call_count = {"expensive": 0}

def expensive_computation(item_id: int) -> dict:
    call_count["expensive"] += 1
    time.sleep(0.01)  # Simulate slow operation
    return {"id": item_id, "result": item_id ** 2, "computed_at": datetime.now().isoformat()}

@app3.get("/cached/{item_id}")
def get_cached(item_id: int):
    key = f"item:{item_id}"
    cached = cache.get(key)
    if cached:
        return {**cached, "cache": "HIT"}
    result = expensive_computation(item_id)
    cache.set(key, result)
    return {**result, "cache": "MISS"}

client3 = TestClient(app3)

# First call — cache miss
r = client3.get("/cached/7")
print("Call 1:", r.json())

# Second call — cache hit
r = client3.get("/cached/7")
print("Call 2:", r.json())

# Different key — cache miss
r = client3.get("/cached/3")
print("Call 3 (new key):", r.json())

print(f"expensive_computation called {call_count['expensive']} time(s) (expected 2)")

In [ ]:
# --- Cursor-based pagination ---
from fastapi import FastAPI, Query
from fastapi.testclient import TestClient
import base64
import json

app4 = FastAPI()

# Simulated dataset
PRODUCTS = [{"id": i, "name": f"Product {i}", "price": round(i * 1.5, 2)} for i in range(1, 51)]

def encode_cursor(last_id: int) -> str:
    return base64.b64encode(json.dumps({"last_id": last_id}).encode()).decode()

def decode_cursor(cursor: str) -> int:
    data = json.loads(base64.b64decode(cursor.encode()).decode())
    return data["last_id"]

@app4.get("/products")
def list_products(cursor: str = None, limit: int = Query(5, ge=1, le=20)):
    last_id = decode_cursor(cursor) if cursor else 0
    page = [p for p in PRODUCTS if p["id"] > last_id][:limit]
    next_cursor = encode_cursor(page[-1]["id"]) if len(page) == limit else None
    return {
        "items": page,
        "next_cursor": next_cursor,
        "has_more": next_cursor is not None
    }

client4 = TestClient(app4)

# Page 1
r = client4.get("/products?limit=5")
page1 = r.json()
print("Page 1:", [p["id"] for p in page1["items"]], "| has_more:", page1["has_more"])

# Page 2 using cursor
r = client4.get(f"/products?limit=5&cursor={page1['next_cursor']}")
page2 = r.json()
print("Page 2:", [p["id"] for p in page2["items"]], "| has_more:", page2["has_more"])

# Page 3
r = client4.get(f"/products?limit=5&cursor={page2['next_cursor']}")
page3 = r.json()
print("Page 3:", [p["id"] for p in page3["items"]])

In [ ]:
# --- Server-Sent Events (SSE) concept ---
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
import asyncio
import json as json_lib

app5 = FastAPI()

async def event_generator(topic: str, count: int = 5):
    """Yields SSE-formatted events."""
    for i in range(1, count + 1):
        event_data = json_lib.dumps({"topic": topic, "event": i, "message": f"Update {i} for {topic}"})
        # SSE format: 'data: <payload>\n\n'
        yield f"id: {i}\n"
        yield f"event: update\n"
        yield f"data: {event_data}\n\n"
        await asyncio.sleep(0)
    # Send a final 'done' event
    yield f"event: done\ndata: {{\"topic\": \"{topic}\", \"status\": \"complete\"}}\n\n"

@app5.get("/events/{topic}")
def sse_stream(topic: str, count: int = 5):
    return StreamingResponse(
        event_generator(topic, count),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no"  # Disable nginx buffering
        }
    )

client5 = TestClient(app5)

r = client5.get("/events/prices?count=3")
print("SSE Content-Type:", r.headers.get("content-type"))
print("SSE raw output:")
print(r.text)

# Parse SSE events manually
print("Parsed events:")
for block in r.text.strip().split("\n\n"):
    lines = block.strip().split("\n")
    event = {}
    for line in lines:
        if ": " in line:
            key, val = line.split(": ", 1)
            event[key] = val
    if event:
        print(" ", event)